In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from urllib.parse import urljoin

def crawl_jobkorea_data():
    
    # 검색 URL
    url = "https://www.jobkorea.co.kr/Search/?stext=%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%EC%84%9D&Page_No=1"
    
    # 헤더 설정 (실제 브라우저처럼 보이게)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ko-KR,ko;q=0.9,en;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }
    
 
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    job_data = []

    job_list = []
    
    title_links = soup.find_all('a', href=lambda href: href and '/Recruit/GI_Read/' in href)
    print(f"제목 링크 수: {len(title_links)}")
    
    processed_urls = set() 
    
    for link in title_links:
        href = link.get('href', '')
        if href in processed_urls:
            continue
        processed_urls.add(href)
        
        current = link
        job_container = None
        
        for level in range(7): 
            current = current.parent
            if current is None:
                break
            
            if current.name in ['div', 'li', 'article']:
                company_check = current.find('span', class_=lambda x: x and 'Typography_variant_size16' in str(x))
                detail_check = current.find('div', class_=lambda x: x and 'Flex_gap_space16' in str(x))
                
                if company_check and detail_check:
                    job_container = current
                    break
                elif level >= 4: 
                    job_container = current
                    break
        
        if job_container:
            job_list.append(job_container)
    
    print(f"찾은 채용공고 컨테이너 수: {len(job_list)}")
    
    for idx, job_container in enumerate(job_list, 1):
        company_elem = job_container.find('a', style=lambda style: style and 'max-width:120px' in style)
        if not company_elem:
            company_elem = job_container.find('span', class_=lambda x: x and 'Typography_variant_size16' in str(x))
        
        company = company_elem.get_text(strip=True) if company_elem else "회사명 없음"
        
        title_elem = job_container.find('a', class_='h7nnv12')
        if title_elem:
            title_span = title_elem.find('span', class_=lambda x: x and 'Typography_variant_size18' in str(x) and 'Typography_truncate' in str(x))
            if title_span:
                title = title_span.get_text(strip=True)
            else:
                title = title_elem.get_text(strip=True)
            
            job_url = urljoin("https://www.jobkorea.co.kr", title_elem['href'])
        else:
            title = "제목 없음"
            job_url = "URL 없음"
        
        detail_div = job_container.find('div', class_=lambda x: x and 'Flex_gap_space16' in str(x))
        if detail_div:
            detail_spans = detail_div.find_all('span', class_=lambda x: x and 'Typography_variant_size14' in str(x) and 'Typography_color_gray800' in str(x))
            details = [span.get_text(strip=True) for span in detail_spans if span.get_text(strip=True)]
            detail = ' | '.join(details) if details else "상세정보 없음"
        else:
            all_spans = job_container.find_all('span')
            details = []
            for span in all_spans:
                text = span.get_text(strip=True)
                if (text and text not in [title, company] and 
                    any(keyword in text for keyword in ['경력', '학력', '정규직', '계약직', '인턴', '구', '시', '월', '일'])):
                    details.append(text)
            detail = ' | '.join(details[:5]) if details else "상세정보 없음"  # 최대 5개만
        
        job_data.append({
            'Site': 'Job_Korea',
            'Col_company': company,
            'Col_Recruit': title,
            'Col_detail': detail,
            'Col_url': job_url
        })
        
        print(f"{idx}. {company} - {title}")
            
        
    

    df = pd.DataFrame(job_data)
    return df
    
            


def save_to_csv(df, folder="data_tmp", filename="jobkorea_data_analysis.csv"):

    if not df.empty:
        if not os.path.exists(folder):
            os.makedirs(folder)
            print(f"'{folder}' 폴더를 생성했습니다.")
        
        filepath = os.path.join(folder, filename)
    
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        print(f"데이터가 '{filepath}' 파일로 저장되었습니다.")
    else:
        print("저장할 데이터가 없습니다.")

def main():
    df = crawl_jobkorea_data()
    save_to_csv(df)
  
    
if __name__ == "__main__":
    main()

제목 링크 수: 40
찾은 채용공고 컨테이너 수: 20
1. 스피치로그㈜ - 신입/경력직데이터분석, 데이터컨설팅,분석보고 작성 및 PT
2. 네이버파이낸셜 - [네이버페이] 대쉬보드(데이터분석) 플랫폼 운영 담당자 (계약직)
3. 더치트주식회사 - 데이터 애널리스트 /데이터분석전문가 / 통계 전문가 / Data Analyst (통
4. 연이 - 데이터분석준전문가 자격증 온라인 강의 교사 채용
5. ㈜지바이크 - [지쿠] 전략마케팅팀데이터분석전문가 경력 채용
6. 콘센트릭스서비스코리아(유) - [Catalyst]데이터분석인턴
7. ㈜트리노드(TreenodInc.) - [서울/경력 7년 이상]데이터분석팀- 데이터 분석 리더
8. ㈜모모랩스 - [모모랩스]데이터분석팀 채용
9. 한패스㈜ - [한패스(주)/데이터분석팀] Google Analytics데이터분석가
10. 휴먼교육센터 - [국비무료/숙식무료]AI/빅데이터분석/풀스택/KDT심화(무료숙식제공)
11. 제이투케이 - 파이썬 인공지능데이터분석으로 퀀트 개발하실 분 구합니다
12. 롯데손해보험㈜ - 장기상품개발, 재무회계,데이터분석, 일반보험 상품개발/UW 경력채용
13. ㈜리만코리아 - [서울] 영업기획 및데이터분석(주임-대리)
14. ㈜이젠아카데미 - SQLD/ADSP/FDA자격증 빅데이터분석평일/주말(토요일) 강사 채용
15. 메가스터디컴퓨터아카데미학원 - [메가스터디IT] 파이썬,데이터분석강사 채용 (방학특강)
16. 두산에너빌리티㈜ - 2025 채용연계형 인턴십 -데이터분석기반 AI 모델 개발
17. ㈜커리어게이트 - [메가스터디IT] 파이썬,데이터분석, ai기초 강사 채용
18. ㈜엑셀리언트 - (삼성동) 사업 및 영업관리자 모집 (디지털 마케팅,데이터분석회사)
19. ㈜네피리티 - [(주)네피리티/서울]데이터분석및 AI모델 개발 모집
20. 팀스파르타㈜ - [기업교육] 실시간 강의 튜터 (Python,데이터분석, AI 활용)
데이터가 'data_tmp/jobkorea_data_analysis.csv' 